In [ ]:
import pandas as pd
import re
import sys
from pathlib import Path
import numpy as np
import importlib
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, median_absolute_error

In [ ]:
BASE_DIR = Path.cwd().parent

sys.path.insert(0, str(BASE_DIR / "src"))

STAGE2_TRAIN_PATH = BASE_DIR / "data" / "processed" / "stage2" / "stage2_train_df.csv"
STAGE2_VAL_PATH   = BASE_DIR / "data" / "processed" / "stage2" / "stage2_val_df.csv"

print("Current directory:", Path.cwd())
print("Stage 2 TRAIN file:", STAGE2_TRAIN_PATH.resolve(), "| exists:", STAGE2_TRAIN_PATH.exists())
print("Stage 2 VAL file:  ", STAGE2_VAL_PATH.resolve(),   "| exists:", STAGE2_VAL_PATH.exists())

Current directory: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\notebooks
Stage 2 TRAIN file: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage2\stage2_train_df.csv | exists: True
Stage 2 VAL file:   E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage2\stage2_val_df.csv | exists: True


In [ ]:
for p in (STAGE2_TRAIN_PATH, STAGE2_VAL_PATH):
    if not p.exists():
        raise FileNotFoundError(f"Stage 2 file not found: {p.resolve()}")

stage2_train_df = pd.read_csv(STAGE2_TRAIN_PATH)
stage2_val_df   = pd.read_csv(STAGE2_VAL_PATH)

print("TRAIN shape:", stage2_train_df.shape)
print("VAL shape:  ", stage2_val_df.shape)
# same columns, but the column ORDER differs between the two files -> compare as sets and align val to train
assert set(stage2_train_df.columns) == set(stage2_val_df.columns), "train/val columns differ"
stage2_val_df = stage2_val_df[stage2_train_df.columns]
print("Columns:", stage2_train_df.columns.tolist())

display(stage2_train_df.head())

TRAIN shape: (5165, 60)
VAL shape:   (1252, 60)
Columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'iphone_model', 'samsung_model', 'is_phone_listing', 'xiaomi_model', 'oppo_model', 'honor_model', 'realme_model', 'infinix_model', 'vivo_model', 'huawei_model', 'motorola_model', 'oneplus_model', 'zte_model', 'tecno_model', 'itel_model', 'google_model', 'sony_model', 'nokia_model', 'lenovo_model', 'lg_model', 'model_key', 'match_rule', 'confidence', 'family', 'storage_known', 'ram_known', 'condition_known']


,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,nokia_model,lenovo_model,lg_model,model_key,match_rule,confidence,family,storage_known,ram_known,condition_known
0,ايفون 13 iPhone,19500.0,مساكن المهندسين، مدينة نصر,/ad/%D8%A7%D9%8A%D9%81%D9%88%D9%86-13-iphone-I...,208854767,legacy,apple,NaN,NaN,NaN,...,NaN,NaN,NaN,apple iphone 13,apple:numeric_pattern,1.0,iphone,False,False,False
1,Honor Magic V5,80000.0,اوبن اير مول، مدينتي,/ad/honor-magic-v5-ID209010621.html,209010621,legacy,honor,NaN,NaN,NaN,...,NaN,NaN,NaN,honor magic v5,honor:rule_05,NaN,magic,False,False,False
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,57999.0,سموحة، الإسكندرية,/ad/oppo-find-x9-pro-%D8%B6%D9%85%D8%A7%D9%86-...,209003975,legacy,oppo,NaN,NaN,NaN,...,NaN,NaN,NaN,oppo find x9 pro,oppo:rule_03,NaN,find,False,False,False
3,Honor Robot Phone 512G 12G ram,94000.0,مدينة نصر، القاهرة,/ad/honor-robot-phone-512g-12g-ram-ID208841945...,208841945,legacy,honor,512.0,12.0,L1:unit,...,NaN,NaN,NaN,honor robot,honor:rule_11,NaN,robot,True,True,False
4,شاومي 17 الترا xiaomi 17 ultra,70000.0,التجمع الخامس، القاهرة الجديدة,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-17-%D8%A7%D...,208281312,legacy,xiaomi,NaN,NaN,NaN,...,NaN,NaN,NaN,xiaomi 17 ultra,xiaomi:rule_17,NaN,xiaomi,False,False,False


## Protocol: which file is used for what
- **TRAIN** (`stage2_train_df`) → everything we *develop* with: GroupKFold CV, comparing experiments, choosing `alpha`.
- **VAL** (`stage2_val_df`) → a **hold-out**: touched only at the end, once, with the final settings frozen. It is never used for CV, tuning or choosing features.
- Models are *fit on all of TRAIN* and *scored on VAL*. Putting train and val back together would remove the only unbiased estimate we have.

In [ ]:
# Sanity checks: the two files must not share ads or title groups, otherwise VAL is not a real hold-out
for key in ["ad_id", "ad_url", "title_group"]:
    overlap = set(stage2_train_df[key].dropna()) & set(stage2_val_df[key].dropna())
    print(f"{key:12s} overlap train/val: {len(overlap)}")
    assert len(overlap) == 0, f"LEAKAGE: {key} appears in both train and val"

ad_id        overlap train/val: 0
ad_url       overlap train/val: 0
title_group  overlap train/val: 0


# Baseline

A baseline is a simple method used to predict prices. Its purpose is to answer:

How well can we predict phone prices using a simple approach, before introducing more complex models?

We need a starting point to determine whether a new model actually improves prediction accuracy.

to understand which information contributes most to price prediction, and make sure that future improvements are supported by evidence.

In [ ]:
def prepare_baseline_df(df):
    """Same filter as before: keep rows with a usable price and title_group."""
    out = df.loc[df["price"].notna() & df["title_group"].notna()].copy()
    out["price"] = pd.to_numeric(out["price"], errors="coerce")
    return out.dropna(subset=["price", "title_group"])

baseline_train_df = prepare_baseline_df(stage2_train_df)   # development (CV + tuning)
baseline_val_df   = prepare_baseline_df(stage2_val_df)     # hold-out, used only in the final section

gkf = GroupKFold(n_splits=5)

print("TRAIN rows:", len(baseline_train_df), "| unique groups:", baseline_train_df["title_group"].nunique())
print("VAL rows:  ", len(baseline_val_df),   "| unique groups:", baseline_val_df["title_group"].nunique())
print("Number of folds:", gkf.get_n_splits())

TRAIN rows: 5165 | unique groups: 2240
VAL rows:   1252 | unique groups: 560
Number of folds: 5


In [ ]:
print("train:", baseline_train_df.shape)
print("val:  ", baseline_val_df.shape)

train: (5165, 60)
val:   (1252, 60)


In [ ]:
from pathlib import Path
import sys


In [ ]:
PROJECT_PATH = Path.cwd().parent
sys.path.insert(0, str(PROJECT_PATH/"src"))


In [ ]:
import evaluation.baselines as baselines

importlib.reload(baselines)
from evaluation.baselines import (
    grouped_median_baseline
)

In [ ]:
# ---- Development: grouped 5-fold CV on TRAIN only ----
# E0 - Global Median
E0_predictions, E0_scores = grouped_median_baseline(data=baseline_train_df,group_col=None,prediction_col="pred_E0",n_splits=5)

# E1 - Brand Median
E1_predictions, E1_scores = grouped_median_baseline(data=baseline_train_df,group_col="brand",prediction_col="pred_E1",n_splits=5)

# E2 - Model Median
E2_predictions, E2_scores = grouped_median_baseline(data=baseline_train_df,group_col="model_key",prediction_col="pred_E2",n_splits=5)

# E3 - Family Median
E3_predictions, E3_scores = grouped_median_baseline(data=baseline_train_df,group_col="family",prediction_col="pred_E3",n_splits=5)

In [ ]:
# CV scores on TRAIN
baseline_scores = pd.DataFrame([
    {"Experiment": "E0 - Global Median", **E0_scores},
    {"Experiment": "E1 - Brand Median", **E1_scores},
    {"Experiment": "E2 - Model Median", **E2_scores},
    {"Experiment": "E3 - Family Median", **E3_scores}
])

baseline_scores = baseline_scores.sort_values(by="mae").reset_index(drop=True)

display(baseline_scores)

,Experiment,mae,median_ae
0,E2 - Model Median,6118.268441,3000.0
1,E3 - Family Median,14780.975895,8500.0
2,E1 - Brand Median,15518.445886,9100.0
3,E0 - Global Median,16852.486350,10000.0


# Tf-IDF

to convert phone-ad titles into numerical features that a machine learning model can understand.

then use Ridge Regression to learn relationships between these numerical text features and phone prices.

In [ ]:
import importlib
import evaluation.baselines as baselines
importlib.reload(baselines)

<module 'evaluation.baselines' from 'e:\\DEPI-Grad\\DEPI-GRADUATION-PROJECT\\src\\evaluation\\baselines.py'>

In [ ]:
alpha_values = [0.1, 1, 10, 100, 1000]
alpha_results = []

for alpha in alpha_values:
    print(f"\nTesting alpha = {alpha}")

    _, scores = baselines.grouped_tfidf_ridge_baseline(
        data=baseline_train_df,
        alpha=alpha
    )

    alpha_results.append({
        "Alpha": alpha,
        "MAE": scores["tfidf_ridge_mae"],
        "MedianAE": scores["tfidf_ridge_median_ae"]
    })

alpha_results_df = pd.DataFrame(alpha_results)

alpha_results_df = alpha_results_df.sort_values(
    by="MAE"
).reset_index(drop=True)
best_alpha = alpha_results_df.iloc[0]["Alpha"]
print("Best Alpha:", best_alpha)


Testing alpha = 0.1
Fold 1: MAE = 7502.52
Fold 2: MAE = 9880.66
Fold 3: MAE = 6246.41
Fold 4: MAE = 6713.33
Fold 5: MAE = 8246.35

Testing alpha = 1
Fold 1: MAE = 6432.95
Fold 2: MAE = 10373.84
Fold 3: MAE = 5694.23
Fold 4: MAE = 6529.32
Fold 5: MAE = 7603.63

Testing alpha = 10
Fold 1: MAE = 7409.44
Fold 2: MAE = 11549.86
Fold 3: MAE = 6813.70
Fold 4: MAE = 7438.82
Fold 5: MAE = 9523.64

Testing alpha = 100
Fold 1: MAE = 10274.27
Fold 2: MAE = 15272.72
Fold 3: MAE = 11065.01
Fold 4: MAE = 8975.80
Fold 5: MAE = 15325.82

Testing alpha = 1000
Fold 1: MAE = 13787.42
Fold 2: MAE = 20303.61
Fold 3: MAE = 15220.24
Fold 4: MAE = 11953.17
Fold 5: MAE = 21745.19
Best Alpha: 1.0


In [ ]:
pred_E4_df, tfidf_ridge_scores = (
    baselines.grouped_tfidf_ridge_baseline(
        data=baseline_train_df,
        alpha=best_alpha
    )
)

print(tfidf_ridge_scores)

Fold 1: MAE = 6432.95
Fold 2: MAE = 10373.84
Fold 3: MAE = 5694.23
Fold 4: MAE = 6529.32
Fold 5: MAE = 7603.63
{'tfidf_ridge_mae': 7326.794955186607, 'tfidf_ridge_median_ae': 4341.689262272859}


In [ ]:
# CV scores on TRAIN (development)
baseline_scores = pd.DataFrame([
    {"Experiment": "E0 - Global Median", **E0_scores},
    {"Experiment": "E1 - Brand Median", **E1_scores},
    {"Experiment": "E2 - Model Median", **E2_scores},
    {"Experiment": "E3 - Family Median", **E3_scores},
    {
        "Experiment": "E4 - TF-IDF + Ridge",
        "mae": tfidf_ridge_scores["tfidf_ridge_mae"],
        "median_ae": tfidf_ridge_scores["tfidf_ridge_median_ae"]
    }
])
baseline_scores = baseline_scores.sort_values(by="mae").reset_index(drop=True)
display(baseline_scores)

,Experiment,mae,median_ae
0,E2 - Model Median,6118.268441,3000.000000
1,E4 - TF-IDF + Ridge,7326.794955,4341.689262
2,E3 - Family Median,14780.975895,8500.000000
3,E1 - Brand Median,15518.445886,9100.000000
4,E0 - Global Median,16852.486350,10000.000000


# Final evaluation on the VAL hold-out

 Each model is fit on the whole TRAIN set and scored on VAL.
Groups (brand / model_key / family) that do not exist in TRAIN fall back to the global TRAIN median.

In [ ]:
def holdout_grouped_median(train, val, group_col):
    """Fit group medians on TRAIN, predict VAL. Unseen groups -> global TRAIN median."""
    global_median = train["price"].median()
    if group_col is None:
        pred = np.full(len(val), global_median)
        coverage = 1.0
    else:
        medians = train.groupby(group_col)["price"].median()
        pred = val[group_col].map(medians).fillna(global_median).to_numpy()
        coverage = val[group_col].isin(medians.index).mean()
    return pred, coverage

def holdout_tfidf_ridge(train, val, alpha, text_col="title", ngram_range=(1, 2)):
    """Fit TF-IDF + Ridge on TRAIN, predict VAL. The vectorizer is fit on TRAIN only."""
    vec = TfidfVectorizer(ngram_range=ngram_range)
    X_train = vec.fit_transform(train[text_col].fillna(""))
    X_val = vec.transform(val[text_col].fillna(""))
    model = Ridge(alpha=alpha).fit(X_train, train["price"])
    return model.predict(X_val)

def score(y_true, y_pred):
    return {"mae": mean_absolute_error(y_true, y_pred),
            "median_ae": median_absolute_error(y_true, y_pred)}

In [ ]:

cv_pred = pd.Series(index=baseline_train_df.index, dtype=float)
for tr_idx, te_idx in GroupKFold(n_splits=5).split(baseline_train_df, groups=baseline_train_df["title_group"]):
    cv_pred.iloc[te_idx] = holdout_tfidf_ridge(baseline_train_df.iloc[tr_idx], baseline_train_df.iloc[te_idx], best_alpha)
my_cv_mae = mean_absolute_error(baseline_train_df["price"], cv_pred)
module_cv_mae = tfidf_ridge_scores["tfidf_ridge_mae"]
print(f"TRAIN CV MAE  helper: {my_cv_mae:,.1f}  |  module: {module_cv_mae:,.1f}  |  diff: {abs(my_cv_mae-module_cv_mae):,.1f}")
if abs(my_cv_mae - module_cv_mae) > 0.02 * module_cv_mae:
    print("WARNING: helper and module differ by more than 2% - align holdout_tfidf_ridge with baselines.py")

TRAIN CV MAE  helper: 7,581.2  |  module: 7,326.8  |  diff: 254.4


In [ ]:
y_val = baseline_val_df["price"]
rows = []

for name, col, cv_scores in [
    ("E0 - Global Median", None,        E0_scores),
    ("E1 - Brand Median",  "brand",     E1_scores),
    ("E2 - Model Median",  "model_key", E2_scores),
    ("E3 - Family Median", "family",    E3_scores),
]:
    pred, coverage = holdout_grouped_median(baseline_train_df, baseline_val_df, col)
    s = score(y_val, pred)
    rows.append({"Experiment": name,
                 "CV MAE (train)": cv_scores["mae"],
                 "VAL MAE": s["mae"], "VAL median_ae": s["median_ae"],
                 "VAL rows with seen group": coverage})

pred_e4 = holdout_tfidf_ridge(baseline_train_df, baseline_val_df, best_alpha)
s = score(y_val, pred_e4)
rows.append({"Experiment": "E4 - TF-IDF + Ridge",
             "CV MAE (train)": tfidf_ridge_scores["tfidf_ridge_mae"],
             "VAL MAE": s["mae"], "VAL median_ae": s["median_ae"],
             "VAL rows with seen group": np.nan})

final_scores = pd.DataFrame(rows).sort_values("VAL MAE").reset_index(drop=True)
display(final_scores)

,Experiment,CV MAE (train),VAL MAE,VAL median_ae,VAL rows with seen group
0,E2 - Model Median,6118.268441,6167.183307,3000.000000,0.908147
1,E4 - TF-IDF + Ridge,7326.794955,8349.873600,5039.415261,NaN
2,E3 - Family Median,14780.975895,12632.953674,9000.000000,0.997604
3,E1 - Brand Median,15518.445886,14432.372204,9500.000000,0.943291
4,E0 - Global Median,16852.486350,15422.150160,10500.000000,1.000000


**How to read this table**
- `CV MAE (train)` and `VAL MAE` should be close. A much worse VAL means the CV was optimistic (e.g. near-duplicate ads leaking across folds).
- `VAL rows with seen group` shows how many VAL ads belong to a brand / model / family that exists in TRAIN; for `model_key` the rest fall back to the global median.